In [0]:
dbutils.library.restartPython()

In [0]:
from common_utils.generic import rename_columns , drop_columns , casting
import pyspark.sql.functions as f
from delta.tables import DeltaTable
from common_utils.logging import get_logger
from common_utils.scd import scd_type1
import json

In [0]:
# =========================================
# 0.Logger
# =========================================
logger = get_logger("b2s_customers-process.ipynb")
logger.info("session_started")

# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
logger.info("setting the configurations")


# =========================================
# 2. Read Config
# =========================================
with open(config_path, "r") as f:
  config = json.load(f) 

# =========================================
# 3. CONFIG Sections
# =========================================
source_config = config['source']
target_config = config['target']
write_options_config = config['write_options']
drop_columns_config = config['drop_columns']
condition_config  = config["merge_condition"]

logger.info("Configuration has been done")
#table_name = f'{source_config['base_path']}{source_config['folder_path']}'
table_name = f"{config['source']['base_path']}{config['source']['folder_path']}"
print(table_name)
df = spark.read.table(table_name)
logger.info("reading from the table path %s",table_name)
df.display()

def silver_transformations(df):
    import pyspark.sql.functions as f

    # Customer name should be lowercase
    df = (
        df.withColumn(
            "customer_name",
            f.lower("customer_name")
        )
        .withColumn(
            "customer_name",
            f.split(f.col("customer_name"), ",")
        )
        .withColumn(
            "first_name",
            f.expr("get(customer_name, 1)")
        )
        .withColumn(
            "last_name",
            f.expr("get(customer_name, 0)")
        )
        .withColumn(
            "city",
            f.upper(f.col("city"))
        )
        .withColumn(
            "postcode",
            f.regexp_replace(
                f.col("postcode"),
                r"\.0$",
                ""
            )
        )
        .withColumn(
            "valid_from",
            f.from_unixtime(
                f.expr("try_cast(valid_from as bigint)")
            )
        )
        .withColumn(
            "valid_to",
            f.from_unixtime(
                f.expr("try_cast(valid_to as bigint)")
            )
        )
        .withColumn(
            "country",
            f.lit("USA")
        )
        
    )
    
    drop_columns(df,drop_columns_config) # function drop_columns called

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"
    )

df = spark.read.table(table_name)
cleaned_df = silver_transformations(df)
cleaned_df.display()

target_table =f"{config['target']['base_path']}{config['target']['target_path']}"
condition = condition_config
if spark.catalog.tableExists("target_path"):
    print("Table already Exists --> now updating and inseting records")
    #load data
    target = DeltaTable.forName(spark,target_table).alias("target")
    #SCD type 1
    #Match in source and target --> key exist in target
    (
        target
        .merge(
            cleaned_df.alias("source"),
           condition
        )
        #when matched
        .whenMatchedUpdateAll()

        #when not matched

        .whenNotMatchedInsertAll()

        .execute()

    )
    
else:
    print("tabel does not exist --> now creating the Table")
    spark.sql("""Create SCHEMA if not exists retail_data_platform.silver""")

    cleaned_df.write.format(write_options_config['format']).mode(write_options_config['mode']).saveAsTable(target_table)

In [0]:



def silver_transformations(df):
    import pyspark.sql.functions as f

    # Customer name should be lowercase
    df = (
        df.withColumn(
            "customer_name",
            f.lower("customer_name")
        )
        .withColumn(
            "customer_name",
            f.split(f.col("customer_name"), ",")
        )
        .withColumn(
            "first_name",
            f.expr("get(customer_name, 1)")
        )
        .withColumn(
            "last_name",
            f.expr("get(customer_name, 0)")
        )
        .withColumn(
            "city",
            f.upper(f.col("city"))
        )
        .withColumn(
            "postcode",
            f.regexp_replace(
                f.col("postcode"),
                r"\.0$",
                ""
            )
        )
        .withColumn(
            "valid_from",
            f.from_unixtime(
                f.expr("try_cast(valid_from as bigint)")
            )
        )
        .withColumn(
            "valid_to",
            f.from_unixtime(
                f.expr("try_cast(valid_to as bigint)")
            )
        )
        .withColumn(
            "country",
            f.lit("USA")
        )
        
    )
    removing_columns = ["file_path", "customer_name"] 
    # here i stored the names of the columns I want to drop
    drop_columns(df,removing_columns) # function drop_columns called

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"
    )

df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")
cleaned_df = silver_transformations(df)
cleaned_df.display()

target_table = "retail_data_platform.silver.customers"
condition = "target.customer_id = source.customer_id"
if spark.catalog.tableExists("retail_data_platform.silver.customers"):
    print("Table already Exists --> now updating and inseting records")
    #load data
    target = DeltaTable.forName(spark,target_table).alias("target")
    #SCD type 1
    #Match in source and target --> key exist in target
    (
        target
        .merge(
            cleaned_df.alias("source"),
           condition
        )
        #when matched
        .whenMatchedUpdateAll()

        #when not matched

        .whenNotMatchedInsertAll()

        .execute()

    )
    
else:
    print("tabel does not exist --> now creating the Table")
    spark.sql("""Create SCHEMA if not exists retail_data_platform.silver""")
    cleaned_df.write.format("delta").mode("overwrite").saveAsTable(target_table)


In [0]:
from common_utils.scd import scd_type1
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")
source = silver_transformations(df)
target_table = "retail_data_platform.silver.silver_customers"
condition = "target.customer_id = source.customer_id"

scd_type1(spark,source,target_table,condition)